Dataset link: https://www.kaggle.com/datasets/stefanoleone992/tripadvisor-european-restaurants

In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv("tripadvisor_european_restaurants.csv")
df.head(3)

C:\Users\menes\AppData\Local\Temp\ipykernel_13372\373148343.py:1: DtypeWarning: Columns (0: region) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("tripadvisor_european_restaurants.csv")


,restaurant_link,restaurant_name,original_location,country,region,province,city,address,latitude,longitude,...,excellent,very_good,average,poor,terrible,food,service,value,atmosphere,keywords
0,g10001637-d10002227,Le 147,"[""Europe"", ""France"", ""Nouvelle-Aquitaine"", ""Ha...",France,Nouvelle-Aquitaine,Haute-Vienne,Saint-Jouvent,"10 Maison Neuve, 87510 Saint-Jouvent France",45.961674,1.169131,...,2.0,0.0,0.0,0.0,0.0,4.0,4.5,4.0,NaN,NaN
1,g10001637-d14975787,Le Saint Jouvent,"[""Europe"", ""France"", ""Nouvelle-Aquitaine"", ""Ha...",France,Nouvelle-Aquitaine,Haute-Vienne,Saint-Jouvent,"16 Place de l Eglise, 87510 Saint-Jouvent France",45.957040,1.205480,...,2.0,2.0,1.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN
2,g10002858-d4586832,Au Bout du Pont,"[""Europe"", ""France"", ""Centre-Val de Loire"", ""B...",France,Centre-Val de Loire,Berry,Rivarennes,"2 rue des Dames, 36800 Rivarennes France",46.635895,1.386133,...,3.0,1.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1083397 entries, 0 to 1083396
Data columns (total 42 columns):
 #   Column                             Non-Null Count    Dtype  
---  ------                             --------------    -----  
 0   restaurant_link                    1083397 non-null  str    
 1   restaurant_name                    1083396 non-null  str    
 2   original_location                  1083397 non-null  str    
 3   country                            1083397 non-null  str    
 4   region                             1033074 non-null  str    
 5   province                           742765 non-null   str    
 6   city                               682712 non-null   str    
 7   address                            1083397 non-null  str    
 8   latitude                           1067607 non-null  float64
 9   longitude                          1067607 non-null  float64
 10  claimed                            1081555 non-null  str    
 11  awards                             

In [4]:
# 1. Standardize and filter by city
# Using .str.contains() with case=False handles 'Munich', 'munich', or even trailing spaces safely.
munich_df = df[df['city'].str.contains('munich', case=False, na=False)].copy()

# 2. Verify the new shape and size
print(f"Original dataset size: {df.shape[0]:,} rows")
print(f"Munich dataset size: {munich_df.shape[0]:,} rows")

# 3. Optional: Reset the index for the new dataframe
munich_df.reset_index(drop=True, inplace=True)

Original dataset size: 1,083,397 rows
Munich dataset size: 3,522 rows


In [5]:
munich_df.to_csv('munich_df.csv', index=False)

In [6]:
# 2. Keep rows where cuisines AND at least one descriptive text field exist
# This ensures you don't accidentally wipe out 75% of your dataset
kg_df = munich_df[
    munich_df['cuisines'].notna() & 
    (munich_df['atmosphere'].notna() | munich_df['top_tags'].notna() | munich_df['keywords'].notna())
].copy()

# 3. Clean up the comma-separated strings into uniform Python lists
def clean_tags(text):
    if pd.isna(text):
        return []
    return [tag.strip().lower() for tag in text.split(',')]

kg_df['cuisine_list'] = kg_df['cuisines'].apply(clean_tags)
kg_df['tag_list'] = kg_df['top_tags'].apply(clean_tags)
kg_df['keyword_list'] = kg_df['keywords'].apply(clean_tags)

print(f"Ready for KG processing: {kg_df.shape[0]} restaurants.")

Ready for KG processing: 2928 restaurants.


In [7]:
kg_df.info()

<class 'pandas.DataFrame'>
Index: 2928 entries, 0 to 3521
Data columns (total 45 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   restaurant_link                    2928 non-null   str    
 1   restaurant_name                    2928 non-null   str    
 2   original_location                  2928 non-null   str    
 3   country                            2928 non-null   str    
 4   region                             2928 non-null   str    
 5   province                           2928 non-null   str    
 6   city                               2928 non-null   str    
 7   address                            2928 non-null   str    
 8   latitude                           2916 non-null   float64
 9   longitude                          2916 non-null   float64
 10  claimed                            2928 non-null   str    
 11  awards                             757 non-null    str    
 12  populari

In [10]:
# 1. Clean the column by splitting on commas to create true Python lists
# We use .str.split(', ') which automatically handles the comma splitting
cuisine_lists = kg_df['cuisines'].dropna().str.split(',')

# 2. Explode the lists into individual rows
flat_cuisines = cuisine_lists.explode()

# 3. Strip any trailing or leading whitespaces from individual elements
flat_cuisines = flat_cuisines.astype(str).str.strip()

# 4. Filter out any empty strings that might have resulted from trailing commas
flat_cuisines = flat_cuisines[flat_cuisines != '']

# 5. Calculate total unique count
unique_count = flat_cuisines.nunique()
print(f"✨ Total Unique Cuisines: {unique_count}\n")

# 6. Display the unique values alphabetically
print("📋 Unique Cuisines List:")
print(sorted(flat_cuisines.unique()))

✨ Total Unique Cuisines: 113

📋 Unique Cuisines List:
['Afghani', 'African', 'Albanian', 'American', 'Arabic', 'Argentinian', 'Asian', 'Australian', 'Austrian', 'Balti', 'Bar', 'Barbecue', 'Beer restaurants', 'Beijing cuisine', 'Belgian', 'Brazilian', 'Brew Pub', 'British', 'Cafe', 'Cajun & Creole', 'Calabrian', 'Campania', 'Caribbean', 'Central American', 'Central Asian', 'Central European', 'Central-Italian', 'Chinese', 'Colombian', 'Contemporary', 'Croatian', 'Czech', 'Danish', 'Deli', 'Diner', 'Dining bars', 'Eastern European', 'Egyptian', 'Emilian', 'Ethiopian', 'European', 'Fast food', 'French', 'Fruit parlours', 'Fusion', 'Gastropub', 'Georgian', 'German', 'Greek', 'Grill', 'Hawaiian', 'Healthy', 'Hungarian', 'Indian', 'International', 'Irish', 'Israeli', 'Italian', 'Japanese', 'Japanese Fusion', 'Japanese sweets parlour', 'Korean', 'Latin', 'Lazio', 'Lebanese', 'Lombard', 'Malaysian', 'Mediterranean', 'Mexican', 'Middle Eastern', 'Mongolian', 'Moroccan', 'Native American', 'Nea

In [13]:
cuisine_counts = flat_cuisines.value_counts()

# Convert it to a beautiful summary DataFrame
cuisine_summary = cuisine_counts.to_frame(name='Restaurant Count')
cuisine_summary['Percentage Share (%)'] = (cuisine_summary['Restaurant Count'] / len(df)) * 100

print("📊 Cuisine Distribution Summary (Top 15):")
print("---------------------------------------------")
print(cuisine_summary.head(15))

📊 Cuisine Distribution Summary (Top 15):
---------------------------------------------
                  Restaurant Count  Percentage Share (%)
cuisines                                                
European                       837              0.077257
Italian                        611              0.056397
German                         595              0.054920
Mediterranean                  459              0.042367
Cafe                           453              0.041813
Asian                          366              0.033783
Pizza                          295              0.027229
Central European               274              0.025291
Bar                            239              0.022060
International                  226              0.020860
Fast food                      185              0.017076
Pub                            170              0.015691
Vietnamese                     167              0.015414
American                       130              0.011999
G